In [3]:
class DatasetEntropy:
    kullback_leibler_ref_sample = 0
    kullback_leibler_ref = 0 # entropy relative to a mixture distribution
    kullback_leibler_sample = 0
    jensen_shannon = 0

    def __init__(self, kl, kl_ref, kl_sample, js):
        self.kullback_leibler_ref_sample = kl
        self.kullback_leibler_ref = kl_ref
        self.kullback_leibler_sample = kl_sample
        self.jensen_shannon = js

def support_set(reference, sample):
    support_set = list(np.unique(reference[col]))
    support_set.extend(list(np.unique(sample[col])))
    support_set = list(np.unique(support_set))
    support_set.sort()
    return support_set
        
def mixture_distribution_function(ref_df, sample_df, support_set):
    for df in [ref_df, sample_df]:
        #trials = np.sum(df[dependent_col] * df[col])
        #df['probability'] = df[dependent_col].map(lambda x: x / trials)
        df = df.sort_values(by = [col])
        df = df.reset_index(drop = True)
        
    out_dist = []
    for i in support_set:
        ref_val = 0
        sample_val = 0
        if (len(ref_df[ref_df[col]==i]) > 0):
            ref_val = ref_df[ref_df[col]==i].iloc[0]["frequency"]
        if (len(sample_df[sample_df[col]==i]) > 0):
            sample_val = sample_df[sample_df[col]==i].iloc[0]["frequency"]
        
        decision_var = random.random()
        if (decision_var < 0.5):
            out_dist.append(ref_val)
        else:
            out_dist.append(sample_val)
    return out_dist

def distribution_function_for_support(df, support_set):
    #trials = np.sum(df[dependent_col] * df[col])
    #df['probability'] = df[dependent_col].map(lambda x: x / trials)
        
    out_dist = []
    for i in support_set:
        val = 0
        if (len(df[df[col]==i]) > 0):
            val = df[df[col]==i].iloc[0]["frequency"]
        out_dist.append(val)
    return out_dist

    
def relative_entropy_from_dist(true_dist, model_dist):
    if (len(true_dist) != len(model_dist)):
        print("Incompatible distribution vectors")
        return
    distance = 0
    for i in range(1, len(true_dist), 1):
        if (true_dist[i] != 0):
            if (model_dist[i] == 0):
                #print("Kullback-Leibler divergence is not defined when P(x)!=0 and Q(x)==0")
                #return
                continue
            else:
                kmer_count_distance = true_dist[i] * math.log(true_dist[i] / model_dist[i], 2)
            distance += kmer_count_distance
    return round(distance, 10)
    
def jensen_shannon_divergence(ref_df, sample_df, support_set):    
    mix_dist = mixture_distribution_function(ref_df, sample_df, support_set)
    ref_dist = distribution_function_for_support(ref_df, support_set)
    sample_dist = distribution_function_for_support(sample_df, support_set)
    #ref_count = counts_for_support(ref_df, support_set)
    #sample_count = counts_for_support(sample_df, support_set)
    
    print("mix_dist\t" + str(len(mix_dist)))
    print("ref_dist\t" + str(len(ref_dist)))
    print("sample_dist\t" + str(len(sample_dist)))
    ref_sample_div = relative_entropy_from_dist(ref_dist, sample_dist)
    ref_div = relative_entropy_from_dist(ref_dist, mix_dist)
    sample_div = relative_entropy_from_dist(sample_dist, mix_dist)
    js_div = 0.5 * ref_div + 0.5 * sample_div

    return DatasetEntropy(np.round(ref_sample_div, 3),
                          np.round(ref_div, 3), 
                          np.round(sample_div, 3), 
                          np.round(js_div, 3))

# The entropy of the distribution of all k-mers
def kmer_count_entropy(df):
    entropy = 0
    for index, row in df.iterrows():
        entropy -= row["frequency"] * math.log(row["frequency"], 2)
    return entropy


def plot_js(ref_df, support, samples, axs):
    for i in range(len(samples)):
        sample = samples[i]
        # is the sample drawn from the same distribution as the reference?
        data_entropy = jensen_shannon_divergence(ref_df, sample, support)
    
        print("Jensen-Shannon divergence\t" + str(data_entropy.jensen_shannon))

        mix_dist = mixture_distribution_function(ref_df, sample, support)
        ref_dist = distribution_function_for_support(ref_df, support)
        sample_dist = distribution_function_for_support(sample, support)

        axs[i].plot(support, ref_dist, label = r"$T=$Fly", color = fly_color)
        sample_color = markov_color
        if (names[i]=="Random"):
            sample_color = mason_color
        axs[i].plot(support, sample_dist, label = r"$S=$" + names[i], color = sample_color)
        axs[i].bar(support, mix_dist, label = r"M=$\frac{1}{2}(T+S)$", alpha = 0.25, width = 0.5, color = CB_color_cycle[4])

        #axs[i].text(100, 0.025, r"$D_{KL}(P||Q)=$" + str(data_entropy.kullback_leibler_ref_sample))
        axs[i].text(100, 0.02, r"$D_{KL}(T||M)=$" + str(data_entropy.kullback_leibler_ref))
        axs[i].text(100, 0.015, r"$D_{KL}(S||M)=$" + str(data_entropy.kullback_leibler_sample))
        axs[i].text(100, 0.01, r"JSD=" + str(data_entropy.jensen_shannon))
    
        axs[i].set_xlim(0, 200)
        axs[i].set_ylim(0, 0.055)
        axs[i].set_xlabel(str(k) + "-mer count")
        axs[i].legend()
    
        axs[0].set_ylabel(r"$P(X=x)$")
        
def cumulative_distribution_function(df, lower_bound, upper_bound):
    cum_dist = []
    sum_i = 0

    df = df.sort_values(by = [col])
    df = df.reset_index(drop = True)
    
    min_count = int(df.iloc[0][col])
    if (min_count != np.min(df[col])):
        print("Not sorted")
        return
    #print("df min\t" + str(min_count))
    for i in range(lower_bound, min_count, 1):
        cum_dist.append(0)
    #print("before df min\t" + str(len(cum_dist)))
    if (len(cum_dist) != np.min(df[col]) - 1):
        print("Length mismatch\t")
        print("Gathered val count\t" + str(len(cum_dist)))
        print("Lowest val in df\t" + str(np.min(df[col])))
        return
    
    distance = 0
    #print(df.head())
    for index, row in df.iterrows():
        #print("Index\t" + str(index))
        #print("iloc[index]\t" + str(df.iloc[index][col]))
        if (index > 0):
            distance = int(df.iloc[index][col] - df.iloc[index-1][col])
            #print("dist\t" + str(dist))
            #print("Gathered value count\t" + str(len(cum_dist)))
        sum_i += int(row[dependent_col])
        cum_dist.append(sum_i)
        if (distance > 1):
            for gap in range(distance - 1):
                cum_dist.append(sum_i)
            distance = 0

        if (len(cum_dist) != row[col]):
            print("Length mismatch\t")
            print("Gathered value count\t" + str(len(cum_dist)))
            print("Current val in df\t" + str(int(row[col])))
            return
    
    if (len(cum_dist) != int(np.max(df[col]))):
        print("Length mismatch\t")
        print("Gathered value count\t" + str(len(cum_dist)))
        print("Highest val in df\t" + str(np.max(df[col])))
        return
    #print("after df max\t" + str(len(cum_dist)))
        
    max_count = int(df.iloc[len(df[col])-1][col])
    for i in range(max_count + 1, upper_bound + 1, 1):
        cum_dist.append(sum_i)
    
    return cum_dist

def kolmogorov_smirnov_statistic_abs(a, b): # a and b are cumulative distribution functions
    if (len(a) != len(b)):
        print("Mismatched lengths")
        print("len a\t" + str(len(a)))
        print("len b\t" + str(len(b)))
        return

    i = 0
    max_dist = 0
    while (i < len(a)):
        dist = abs(a[i] - b[i])
        if (dist > max_dist):
            max_dist = dist
        i+=1
    return max_dist

def plot_ks(ref_cdf, lower, upper, count_sum, samples, axs, is_result = True):
    for i in range(len(samples)):
        sample = samples[i]
        # is the sample drawn from the same distribution as the reference?
        print(names)
        axs[i].plot(list(range(lower, upper + 1, 1)), ref_cdf, label = names[2], color = fly_color)
    
        if (count_sum != np.sum(sample["count"])):
            print("k-mer set size mismatch with diff\t" + str(abs(count_sum - np.sum(sample["count"]))))

        sample_cdf = np.array(cumulative_distribution_function(sample, lower, upper)) / np.sum(sample["count"])
        print(sample_cdf)
        ks_abs = kolmogorov_smirnov_statistic_abs(ref_cdf, sample_cdf) 

        sample_color = markov_color
        if (names[i]=="Mason"):
            sample_color = mason_color
        axs[i].plot(list(range(lower, upper + 1, 1)), sample_cdf, label = names[i], color = sample_color)
    
        print("KS abs\t" + str(ks_abs))
        ks_abs / count_sum
    
        max_div_ind = 0
        max_div = 0
        lower_at_div = 0
        higher_at_div = 0
        for pos in range(len(ref_cdf)):
            cur_div = abs(ref_cdf[pos] - sample_cdf[pos])
            if (cur_div > max_div):
                max_div_ind = pos
                max_div = cur_div
                if (ref_cdf[pos] > ref_cdf[pos]):
                    lower_at_div = sample_cdf[pos]
                    higher_at_div = ref_cdf[pos]
                else:
                    lower_at_div = ref_cdf[pos]
                    higher_at_div = sample_cdf[pos]

        alert_color = CB_color_cycle[7]
        if (is_result):
            axs[i].vlines(max_div_ind, lower_at_div-0.01, higher_at_div-0.01, color = alert_color, label = r"$KS$")
        else:
            axs[i].vlines(max_div_ind, lower_at_div, higher_at_div, color = alert_color, label = r"$KS$")

        if (is_result):
            axs[i].text(max_div_ind - 35, lower_at_div + 0.001, r"$KS=$" + str(round(ks_abs, 2)), color = alert_color) 
        else:
            axs[i].text(max_div_ind, lower_at_div, r"$KS=$" + str(round(ks_abs, 2)), color = alert_color) 
        
        axs[i].set_xlabel(str(k) + "-mer count")
        axs[i].set_xlim(0, 200)
        axs[i].legend()

        axs[0].set_ylabel(r"$P(X \leq x)$")

In [1]:
fig = plt.figure(constrained_layout=True)
fig.set_figheight(6)
fig.set_figwidth(10)

axs = fig.subplots(nrows=2, ncols=2, sharex = True)

col = "occurrences"
dependent_col = "count"
fly_color = CB_color_cycle[2]
mason_color = CB_color_cycle[1]
markov_color = CB_color_cycle[3]

ns = [100, 10000]
ps = [0.1, 0.001]
expected_lambdas = []
names = []
for i in range(len(ns)):
    names.append(r"$Bi(n=" + str(ns[i]) + ", p=" + str(ps[i]) + ")$")
    expected_lambdas.append(ns[i] * ps[i])

for i in range(len(ns)):
    names.append(r"$Po(\lambda = $" + str(int(expected_lambdas[i])) + ")")

n = ns[0]
p = ps[0]
expected_lambda = expected_lambdas[0]

ax = axs[0][0]

x = np.arange(poisson.ppf(0.001, expected_lambda), poisson.ppf(0.999, expected_lambda))
y = poisson.pmf(x, expected_lambda)
po = pd.DataFrame({col : y, 'occurrences' : x, 'count' : x * n})
for c in ["count", "occurrences"]:
    po[c] = po[c].astype(int)

ax.plot(x, y, color = "black", label = names[2])

x = np.arange(binom.ppf(0.001, n, p), binom.ppf(0.999, n, p))
y = binom.pmf(x, n, p)
bi1 = pd.DataFrame({col : y, 'occurrences' : x, 'count' : x * n})
for c in ["count", "occurrences"]:
    bi1[c] = bi1[c].astype(int)

ax.plot(x, y, color = "blue", label = names[0])
ax.legend()

ax = axs[0][1]

n = ns[1]
p = ps[1]
expected_lambda = n*p

x = np.arange(poisson.ppf(0.001, expected_lambda), poisson.ppf(0.999, expected_lambda))
y = poisson.pmf(x, expected_lambda)
po = pd.DataFrame({col : y, 'occurrences' : x, 'count' : x * n})
for c in ["count", "occurrences"]:
    po[c] = po[c].astype(int)

ax.plot(x, y, color = "black", label = names[3])

x = np.arange(binom.ppf(0.001, n, p), binom.ppf(0.999, n, p))
y = binom.pmf(x, n, p)
bi2 = pd.DataFrame({col : y, 'occurrences' : x, 'count' : x * n})
for c in ["count", "occurrences"]:
    bi2[c] = bi2[c].astype(int)

ax.plot(x, y, color = "blue", label = names[1])
ax.set_ylabel("$P(X = x)$")

lower = np.min([np.min(po[col]), np.min(bi1[col]), np.min(bi2[col])])
lower = 1
upper = np.max([np.max(po[col]), np.max(bi1[col]), np.max(bi2[col])])

support = support_set(bi1, bi2)
support.extend(support_set(bi1, bi2))
print("Length of support\t" + str(len(support)))

ref_cdf = cumulative_distribution_function(po, lower, upper)
ref_cdf = np.array(ref_cdf) / np.sum(po["count"])

samples = [bi1, bi2]
ks_axs = [axs[1][0], axs[1][1]]
plot_ks(ref_cdf, lower, upper, np.sum(po["count"]), samples, ks_axs, is_result = False)
for a in ks_axs:
    a.set_xlim([0, np.max([np.max(po[col]), np.max(bi1[col]), np.max(bi2[col])]) + 1])

ax.legend()
axs[0][0].set_ylabel("$P(X = x)$")

for ax in axs:
    for a in ax:
        a.set_xlabel("Number of successes")
#fig.tight_layout()
fig.savefig('ks_statistic_results.png', dpi = 200)

NameError: name 'plt' is not defined

In [2]:
kmers = [7, 9, 11, 13, 15]
o=10

color_gradient = ["#4DAF4A", "#4EB36D", "#50B898", "#51BCC0", "#5A93B7", "#6367C1", "#6A40C9"]

fig, ax = plt.subplots(5, 1)
fig.set_figheight(12)
fig.set_figwidth(6)
#fig.suptitle("Markov model fit to the fly reference ")

def run_plot(i, max_count, max_freq):
    j = i
    # plot binomial trial
    # ax.plot(maso_occ["occurrences"], rand_occ["frequency"], color = CB_color_cycle[6], label = "B(n,p)", alpha = 0.5)

    k = kmers[i]
    markov_occ = read_occ_table("data/simulated_fly_size/from_fly_markov_o" + str(o) + ".fa." + str(k) + "mer.tsv")
    ref_occ = read_occ_table("data/fly/dna4.fasta." + str(k) + "mer.tsv")
    #mason_occ = read_occ_table("data/simulated_fly_size/mason.fa." + str(k) + "mer.tsv")    
    max_freq = np.max([np.max(markov_occ["frequency"]), np.max(ref_occ["frequency"])]) * 1.25
    #plot_abundance(ax[j], mason_occ, max_count, max_freq, "Mason", "darkgray")
    plot_abundance(ax[j], ref_occ, max_count, max_freq, "Fly", "gray")
    plot_abundance(ax[j], markov_occ, max_count, max_freq, "Markov o=" + str(o) + "", color_gradient[i])
    
    plot_poisson(ax[j], markov_occ)
        
    ax[j].set_title(str(k) + "-mer frequency spectra of 140Mb sequences")
    ax[j].legend(loc = "upper right")
    ax[j].set_xlabel("K-mer occurrences")
    ax[j].set_ylabel("Frequency")

run_plot(0, 50000, 0.0005)
run_plot(1, 5000, 0.002)
run_plot(2, 500, 0.02)
run_plot(3, 50, 0.2)
run_plot(4, 5, 0.85)
        
#ax.text(95, 0.008, "Total k-mer count: 143M", size = SMALL_SIZE)
#ax.text(95, 0.004, "E(x)=" + str(round(np.sum(markov_occ["count"] * markov_occ["occurrences"]) / np.sum(markov_occ["count"]))), size = XSMALL_SIZE)

#handles, labels = ax1.get_legend_handles_labels()
#fig.legend(handles, labels, bbox_to_anchor=(0.935,0.75))

fig.tight_layout()
fig.savefig('140Mb_spectra_markov_kmers.png', dpi = 200)

NameError: name 'plt' is not defined